In [1]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.metrics import pairwise_distances
from scipy.stats import gaussian_kde
from scipy.spatial import distance
from collections import Counter
import os
from tqdm import tqdm
from itertools import combinations


%matplotlib inline
plt.rcParams['figure.dpi'] = 150

In [2]:
# Distances among descriptors
path_to_pocketvec = '../processed/pocketvec_RUN/'

# Load tRNAs PocketVec descriptors
fps_tRNAs = pickle.load(open("../processed/pocketvec_RUN/fps_rank.pkl", 'rb'))
print(len(fps_tRNAs))

# Get the protein for each PocketVec descriptor
proteins = sorted(set([i.split("_")[1] for i in fps_tRNAs]))
print(len(proteins))

# For each protein, get list of pockets
protein_to_pockets = {}
for p in sorted(set(proteins)):
    protein_to_pockets[p] = [i for i in fps_tRNAs if i.split("_")[1] == p]

276
21


In [3]:
def get_min_pocketvec_distance(fps1, fps2):
    """
    Get the minimum distance between two PocketVec descriptors
    """
    return np.min(pairwise_distances(fps1, fps2, metric='cosine').flatten())   

def get_min_pocketvec_distance_same_protein(fps1, fps2):
    """
    Get the minimum distance between two PocketVec descriptors,
    using only the upper triangle of the distance matrix.
    """
    dists = pairwise_distances(fps1, fps2, metric='cosine')
    triu_indices = np.triu_indices_from(dists, k=1)
    return np.min(dists[triu_indices])  

def binarize_distance(value, threshold=0.15):
    """
    Binaruze distance values
    """
    if value <= threshold:
        return 1
    else:
        return 0

In [4]:
# Load interpro annotations

pocket_detection_interpro_df = pd.read_csv("../processed/pocket_detection_data_interpro.tsv", sep="\t")

# Hide unnecessary columns
del pocket_detection_interpro_df['Pocket centroid coordinate (x y z)']
del pocket_detection_interpro_df['Pocket residues (chain_resn)']
del pocket_detection_interpro_df['B-factors']
del pocket_detection_interpro_df['Full path']
del pocket_detection_interpro_df['Interpro Matches']

# Add "pocket ID" column
pocket_detection_interpro_df['Pocket ID'] = ["_".join([j.strip(".pdb"),f"pocket_{k}"]) 
                                             for i,j,k in zip(pocket_detection_interpro_df['Uniprot AC'], 
                                             pocket_detection_interpro_df['File name'], 
                                             pocket_detection_interpro_df['Pocket number'])]

# Pocket to interpro
pocket_to_interpro = {}
for pocket in sorted(fps_tRNAs):
    annotations = pocket_detection_interpro_df[pocket_detection_interpro_df['Pocket ID'] == pocket]
    annotations = sorted(set(annotations["Interpro curated annotation"].tolist()))
    pocket_to_interpro[pocket] = annotations


# Load sequential comparisons

# NW algorithm
PROP_NW = pd.read_csv("../processed/sequences/NW_SeqAlign/Prop_matrix.tsv", sep='\t', index_col=0)
PROP_NW.columns = PROP_NW.columns.str.replace('(tRNA)', '')
PROP_NW.index = PROP_NW.index.str.replace('(tRNA)', '')
PROP_NW.columns = PROP_NW.columns.str.strip()
PROP_NW.index = PROP_NW.index.str.strip()
PROP_NW = PROP_NW.stack().to_dict()

SEQ_ID_NW = pd.read_csv("../processed/sequences/NW_SeqAlign/SeqId_matrix.tsv", sep='\t', index_col=0)
SEQ_ID_NW.columns = SEQ_ID_NW.columns.str.replace('(tRNA)', '')
SEQ_ID_NW.index = SEQ_ID_NW.index.str.replace('(tRNA)', '')
SEQ_ID_NW.columns = SEQ_ID_NW.columns.str.strip()
SEQ_ID_NW.index = SEQ_ID_NW.index.str.strip()
SEQ_ID_NW = SEQ_ID_NW.stack().to_dict()

# CLUSTAL OMEGA
with open('../processed/sequences/MSA_ClustalOmega/clustalo-I20250414-162824-0073-80384311-p1m.pim') as f:
    lines = f.readlines()
lines = [i for i in lines if i.startswith('#') == False and len(i.strip()) > 0]
clustal_omega = {}
for i, line in enumerate(lines):
    parts = line.split()
    uniprot_1 = parts[1].split('|')[0]
    seq_identities = list(map(float, parts[2:]))
    for j, identity in enumerate(seq_identities):
        uniprot_2 = lines[j].split()[1].split('|')[0]
        clustal_omega[(uniprot_1, uniprot_2)] = identity

# Load structural comparisons
PATH_TO_RMSDs = "../processed/structural_comparisons"
RMSDs = {}
for c, protein1 in enumerate(proteins):
    RMSDs[(protein1, protein1)] = 0
    for protein2 in proteins[c+1:]:
        df = pd.read_csv(os.path.join(PATH_TO_RMSDs, f"{protein1}_{protein2}_rmsd.csv"))
        min_rmsd = min(df["rmsd"])
        RMSDs[(protein1, protein2)] = min_rmsd
        RMSDs[(protein2, protein1)] = min_rmsd

In [6]:
#############
### PAIRS ###
#############

In [7]:
# Prepare pd DF sumarizing ALL results (SEQ, STRUCTURE AND POCKETVEC)

ALL_RESULTS_PAIRS = []

# Load proteins
proteins = sorted(set(pd.read_csv(os.path.join("..", "processed", "pocket_detection_data.csv"))['Uniprot AC']))

# For each pair of proteins
for c,protein1 in tqdm(enumerate(proteins)):
    for protein2 in proteins[c+1:]:
        
        # Get all pockets from all structures
        pockets1 = protein_to_pockets[protein1]
        pockets2 = protein_to_pockets[protein2]

        # For each pair of pockets
        for pocket1 in pockets1:
            for pocket2 in pockets2:

                # Get interpro labels
                interpro_pocket1 = ";".join(pocket_to_interpro[pocket1])
                interpro_pocket2 = ";".join(pocket_to_interpro[pocket2])

                # Get pocketvec distance
                dist = round(distance.cosine(fps_tRNAs[pocket1], fps_tRNAs[pocket2]), 3)

                # Masif placeholder
                # ...

                results = [protein1, pocket1, interpro_pocket1, protein2, pocket2, interpro_pocket2, dist,
                           RMSDs[(protein1, protein2)], SEQ_ID_NW[(protein1, protein2)], clustal_omega[(protein1, protein2)]]

                ALL_RESULTS_PAIRS.append(results)

# Store pandas df
ALL_RESULTS_PAIRS = pd.DataFrame(ALL_RESULTS_PAIRS, columns=["Protein1", "Pocket1", "InterPro-Pocket1", "Protein2", "Pocket2", "InterPro-Pocket2", 'PocketVec distance',
                                                 "Protein RMSD", 'Protein SEQ ID (CO)', 'Protein SEQ ID (NW)'])
ALL_RESULTS_PAIRS = ALL_RESULTS_PAIRS.sort_values('PocketVec distance').reset_index(drop=True)

21it [00:00, 57.67it/s]


In [8]:
ALL_RESULTS_PAIRS.to_csv("../processed/protein_priorization/pairs.tsv", sep='\t', index=False)

In [9]:
################
### TRIPLETS ###
################

In [10]:
len(proteins) ** 3

9261

In [11]:
len([[i,j,k] for i in proteins for j in proteins for k in proteins])

9261

In [12]:
len([tuple(sorted([i,j,k])) for i in proteins for j in proteins for k in proteins])

9261

In [13]:
len([tuple(sorted([i,j,k])) for i in proteins for j in proteins for k in proteins if i != j and i != k and j != k])

7980

In [14]:
len(set([tuple(sorted([i,j,k])) for i in proteins for j in proteins for k in proteins if i != j and i != k and j != k]))

1330

In [15]:
ALL_RESULTS_TRIPLETS = []

# Load proteins
proteins = sorted(set(pd.read_csv(os.path.join("..", "processed", "pocket_detection_data.csv"))['Uniprot AC']))

# Calculate triplets
protein_triplets = list(combinations(proteins, 3))

# For each triplet of proteins
for protein1, protein2, protein3 in tqdm(protein_triplets):

    # Get all pockets from all structures
    pockets1 = protein_to_pockets[protein1]
    pockets2 = protein_to_pockets[protein2]
    pockets3 = protein_to_pockets[protein3]

    # For each pair of pockets
    for pocket1 in pockets1:
        for pocket2 in pockets2:
            for pocket3 in pockets3:

                # Get interpro labels
                interpro_pocket1 = ";".join(pocket_to_interpro[pocket1])
                interpro_pocket2 = ";".join(pocket_to_interpro[pocket2])
                interpro_pocket3 = ";".join(pocket_to_interpro[pocket3])

                # A (1-2), # B (1-3), #C (2-3)
                dist_A = round(distance.cosine(fps_tRNAs[pocket1], fps_tRNAs[pocket2]), 3)
                dist_B = round(distance.cosine(fps_tRNAs[pocket1], fps_tRNAs[pocket3]), 3)
                dist_C = round(distance.cosine(fps_tRNAs[pocket2], fps_tRNAs[pocket3]), 3)

                # Masif placeholder
                # ...

                # RMSDs
                RMSD_A = RMSDs[(protein1, protein2)]
                RMSD_B = RMSDs[(protein1, protein3)]
                RMSD_C = RMSDs[(protein2, protein3)]

                # SEQ ID (CO)
                SEQ_ID_CO_A = clustal_omega[(protein1, protein2)]
                SEQ_ID_CO_B = clustal_omega[(protein1, protein3)]
                SEQ_ID_CO_C = clustal_omega[(protein2, protein3)]

                # SEQ ID (NW)
                SEQ_ID_NW_A = SEQ_ID_NW[(protein1, protein2)]
                SEQ_ID_NW_B = SEQ_ID_NW[(protein1, protein3)]
                SEQ_ID_NW_C = SEQ_ID_NW[(protein2, protein3)]

                results = [protein1, pocket1, interpro_pocket1, 
                            protein2, pocket2, interpro_pocket2, 
                            protein3, pocket3, interpro_pocket3, 
                            dist_A, dist_B, dist_C,
                            RMSD_A, RMSD_B, RMSD_C,
                            SEQ_ID_CO_A, SEQ_ID_CO_B, SEQ_ID_CO_C,
                            SEQ_ID_NW_A, SEQ_ID_NW_B, SEQ_ID_NW_C]

                ALL_RESULTS_TRIPLETS.append(results)


# Store pandas df
ALL_RESULTS_TRIPLETS = pd.DataFrame(ALL_RESULTS_TRIPLETS, columns=["protein1", "pocket1", "interpro_pocket1", 
                                                                    "protein2", "pocket2", "interpro_pocket2", 
                                                                    "protein3", "pocket3", "interpro_pocket3", 
                                                                    "dist_A", "dist_B", "dist_C",
                                                                    "RMSD_A", "RMSD_B", "RMSD_C",
                                                                    "SEQ_ID_CO_A", "SEQ_ID_CO_B", "SEQ_ID_CO_C",
                                                                    "SEQ_ID_NW_A", "SEQ_ID_NW_B", "SEQ_ID_NW_C"])
ALL_RESULTS_TRIPLETS = ALL_RESULTS_TRIPLETS.sort_values(["dist_A", "dist_B", "dist_C"]).reset_index(drop=True)

100%|██████████| 1330/1330 [01:23<00:00, 15.97it/s]


In [16]:
len(set([tuple(sorted([i,j,k])) for i,j,k in zip(ALL_RESULTS_TRIPLETS['protein1'], ALL_RESULTS_TRIPLETS['protein2'], ALL_RESULTS_TRIPLETS['protein3'])]))

1330

In [17]:
len(ALL_RESULTS_TRIPLETS)

2850834

In [18]:
ALL_RESULTS_TRIPLETS.to_csv("../processed/protein_priorization/triplets.tsv", sep='\t', index=False)